# Requirements

In [4]:
import os
import re, unicodedata
from bs4 import BeautifulSoup

### Load newest local Glaux files

In [23]:
greek_data_dir = "../data/sources/"
source_dir = os.path.join(greek_data_dir, "GLAUx_v1-0/xml/")

# Get filenames and corresponding TLG IDs
glaux_filenames = os.listdir(source_dir)
glaux_tlgs = [re.sub(r'(\d{4})-(\d{3}).xml', r'tlg\1.tlg\2', fn) for fn in glaux_filenames]

print(f"There are {len(glaux_filenames)} GLAUx XML files in {source_dir}.\n\nExample filenames: {glaux_filenames[:10]}.\n\nCorresponding TLG IDs: {glaux_tlgs[:10]}\n\n")

with open(source_dir  + glaux_filenames[0], "r") as f:
    soup = BeautifulSoup(f.read())
print(f"Example file content:\n{soup.prettify()[:5000]}")

There are 1421 GLAUx XML files in ../data/sources/GLAUx_v1-0/xml/.

Example filenames: ['0062-032.xml', '0074-001.xml', '0127-001.xml', '0007-134.xml', '0094-002.xml', '1286-010.xml', '0031-018.xml', '0007-002.xml', '0057-020.xml', '0014-034.xml'].

Corresponding TLG IDs: ['tlg0062.tlg032', 'tlg0074.tlg001', 'tlg0127.tlg001', 'tlg0007.tlg134', 'tlg0094.tlg002', 'tlg1286.tlg010', 'tlg0031.tlg018', 'tlg0007.tlg002', 'tlg0057.tlg020', 'tlg0014.tlg034']


Example file content:
<treebank version="2" xml:lang="grc">
 <sentence analysis="auto" document_id="0062-032" id="1" struct_id="671176">
  <word div_section="1" form="Ἑρμῆ" head="113444733" id="113444727" lemma="Ἑρμῆς" postag="n-s---mv-" relation="ExD" speaker="Ζεύς">
  </word>
  <word div_section="1" form="," head="0" id="113444728" lemma="," postag="u--------" relation="AuxX" speaker="Ζεύς">
  </word>
  <word div_section="1" form="λαβὼν" head="113444733" id="113444729" lemma="λαμβάνω" postag="v-sapamn-" relation="ADV" speaker="Ζεύ

### Extract Glaux sentence level data

In [21]:
target_path = "../data/output/GLAUx/pickles"
try:
    os.mkdir(target_path)
except:
    pass

os.listdir(target_path)

[]

In [76]:
# Normalize a string to a specified Unicode form
# keep only basic punctuation (no quotes/brackets)
_PUNCT = r".,;:!?··"

# If you still keep a global class, do NOT include \s (we remove whitespace in the cleaner)
# Also include combining diacritics to be safe.
_CLASS = rf"[^\u0370-\u03FF\u1F00-\u1FFF\u0300-\u036F{re.escape(_PUNCT)}]"

# alternative
# _CLASS = rf"[^Ͱ-Ͽἀ-῿ᾀ-῾\s{re.escape(_PUNCT)}]"

def norm_clean_token(text: str) -> str:
    """NFC, drop controls+ALL whitespace, keep only Greek (incl. combining) + _PUNCT."""
    if not text:
        return ""
    s = unicodedata.normalize("NFC", text)
    # remove controls/format + ALL whitespace
    s = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B\u200C\u200D\u2060\uFEFF\s]+", "", s)
    # allow-list Greek + combining + chosen punctuation
    s = re.sub(_CLASS, "", s)
    return s

In [79]:
POS_REVERSE = {
    "n": "NOUN",
    "v": "VERB",
    "a": "ADJ",
    "r": "ADP",     # but note: 'r' is often used for "adverbs" too in some corpora
    "p": "PRON",
    "l": "DET",     # articles
    "c": "CCONJ",   # or SCONJ; both mapped to 'c'
    "u": "PUNCT",
    "g": "PART",    # e.g. μέν, γάρ etc. → particles
    "z": "X",       # uncategorized, foreign, artificial ellipsis etc.
}

def from_filename_to_sentence_data(fn):
    with open(source_dir + fn, "r") as f:
        soup = BeautifulSoup(f.read(), "xml")
    sentences_data = []
    tlg_doc_id = re.sub(r'(\d{4})-(\d{3})\w?\.xml', r'tlg\1.tlg\2', fn)
    sent_n = 0
    _PUNCT = r".,;:!?··"
    _CLASS = rf"[^Ͱ-Ͽἀ-῿ᾀ-῾\s{re.escape(_PUNCT)}]"
    _HAS_GREEK = re.compile(r"[Ͱ-Ͽἀ-῿ᾀ-῾]")
    for sent in soup.find_all("sentence"):
        sentence = ""
        sent_data = []
        start_index = 0
        n = 0
        for w in sent.find_all("word"):
            token = norm_clean_token(w["form"])
            lemma = norm_clean_token(w.get("lemma", "")) or token
            if len(token) > 0:

                ref = {}
                for attr, val in w.attrs.items():
                    if attr.startswith("div_") or attr == "line":
                        ref[attr] = val

                if n == 0 or w.get("relation") in ["AuxX", "AuxK", "PUNCT"]:
                    start_index = len(sentence)
                    sentence += token
                else:
                    start_index = len(sentence) + 1
                    sentence += " " + token

                end_index = start_index + len(token)
                n += 1
                try:
                    sent_data.append((token, lemma, POS_REVERSE.get(w["postag"][0], "X"), ref, start_index, end_index))
                except (KeyError, IndexError):
                    sent_data.append((token, token, "X", ref, start_index, end_index))
                if w.get("relation") == "AuxK":
                    break
        if len(sent_data) > 0 and any(_HAS_GREEK.search(tok[0]) for tok in sent_data):
            sentences_data.append((tlg_doc_id, sent_n, sentence, sent_data))
            sent_n += 1
    # replace your tlg_doc_id/target_fn lines with this block
    m = re.match(r'^(\d{4})-(\d{3})([a-z])?\.xml$', fn)
    if not m:
        raise ValueError(f"Unexpected filename: {fn}")
    tlg_doc_id = f"tlg{m.group(1)}.tlg{m.group(2)}"  # base id (no letter)
    suffix = m.group(3) or ""                        # "", "a", "b", ...
    target_fn = f"{tlg_doc_id}{suffix}.pickle"       # keep suffix in filename
    with open(target_path + target_fn, "wb") as f:
        pickle.dump(sentences_data, f)

In [80]:
%%time
for fn in glaux_filenames:
    from_filename_to_sentence_data(fn)

CPU times: user 9min 29s, sys: 9.28 s, total: 9min 39s
Wall time: 9min 39s


In [81]:
len(os.listdir(target_path))

1431

In [82]:
import os, re, pickle
from collections import defaultdict

# --- 1) When WRITING per-file pickles, include the optional letter in the name ---
# Use this inside your from_filename_to_sentence_data(fn):
#   m = re.match(r'^(\d{4})-(\d{3})([a-z])?\.xml$', fn)
#   if not m:
#       raise ValueError(f"Unexpected filename: {fn}")
#   suffix = m.group(3) or ""                 # "", "a", "b", ...
#   tlg_doc_id = f"tlg{m.group(1)}.tlg{m.group(2)}"  # base id (no letter)
#   target_fn = f"{tlg_doc_id}{suffix}.pickle"       # keep suffix in filename
#   with open(os.path.join(target_path, target_fn), "wb") as f:
#       pickle.dump(sentences_data, f)

# --- 2) Merge pass: group by base, merge parts in order, write single merged file ---
def merge_letter_parts(target_path: str):
    rx = re.compile(r'^(tlg\d{4}\.tlg\d{3})([a-z])?\.pickle$')
    groups = defaultdict(list)

    for fn in os.listdir(target_path):
        m = rx.match(fn)
        if m:
            base = m.group(1)              # e.g., "tlg0001.tlg001"
            suffix = m.group(2) or ""      # "", "a", "b", ...
            groups[base].append((suffix, fn))

    for base, items in groups.items():
        # order: base (no suffix) first, then a, b, c...
        items.sort(key=lambda t: (t[0] != "", t[0]))
        merged = []
        for suffix, fn in items:
            with open(os.path.join(target_path, fn), "rb") as f:
                merged.extend(pickle.load(f))

        merged_name = f"{base}.pickle"
        merged_path = os.path.join(target_path, merged_name)
        with open(merged_path, "wb") as f:
            pickle.dump(merged, f)

        # delete the part files, keep only the merged file
        for _, fn in items:
            if fn != merged_name:
                os.remove(os.path.join(target_path, fn))

In [84]:
merge_letter_parts(target_path)

In [85]:
len(os.listdir(target_path))

1408

In [86]:
[fn for fn in os.listdir(target_path) if "a" in fn]

[]